# Mosaic AI Agent Framework: Evaluate and deploy a tool-calling LangGraph agent

In the script : agent.py, a tool-calling LangGraph agent wrapped with `ResponsesAgent`.
In this notebook, we will : 

- test your agent locally and look at the traces to understand what it's doing.
- log it to MLflow so you can version it and track experiments.
- create an evaluation dataset with real questions and expected answers.
- run automated evaluation using AI judges to score responses.
-iterate on your design based on the results.
- re-evaluate to measure improvement.
- deploy to production when quality is good enough.
- Test the agent on the playground

To learn more about authoring an agent using Mosaic AI Agent Framework, see Databricks documentation ([AWS](https://docs.databricks.com/aws/generative-ai/agent-framework/author-agent)).




### Load the libraries

In [0]:
%pip install -U -qqqq backoff==2.2.1 databricks-langchain==0.8.2 langgraph==0.6.4 uv databricks-agents==1.9.0 databricks-connect==16.4.7 databricks-langchain==0.8.2 mlflow[databricks]==3.8.1
dbutils.library.restartPython()

### Run Unity Catalog Config

In [0]:
%run ../_config/config_unity_catalog

### Run MLFlow config

In [0]:
%run ../_config/config_agent


## 0 - Agent config
The langgraph agent is already defined and wrapped to integrate Databricks features.

- Agent tools

- Agent retriever 

- System_prompt

- LLM details 

- input sample 


#### Init agent config yaml
As this file is modified during the process.
This cell is used to reset the config to the initial system message and model

In [0]:
import yaml
try:
    config = yaml.safe_load(open("agent_config.yaml"))
    config["config_version_name"] = "meta_version"
    config["system_prompt"] = ("You are an expert customer support agent with access to tools and knowledge bases.\n\n"
                               "## YOUR MISSION\n"
                               "Help customers efficiently by retrieving accurate information and executing actions through the available tools."
    )
    config["llm_endpoint_name"] = "databricks-meta-llama-3-3-70b-instruct"
    yaml.dump(config, open("agent_config.yaml", "w"), default_flow_style=False, allow_unicode=True)
except Exception as e:
    print(f"Skipped update - ignore for job run - {e}")

## 1 - Test the agent

Interact with the agent to test its output and tool-calling abilities. Since this notebook called `mlflow.langchain.autolog()`, you can view the trace for each step the agent takes.

Replace this placeholder input with an appropriate domain-specific example for your agent.

In [0]:
from agent import AGENT
input_example = {"input": [{"role": "user", "content": "what is the order 1003 status ?"}], "custom_inputs": {"session_id": "test-session"}}

prediction = AGENT.predict(input_example)
print(prediction.model_dump(exclude_none=True))

#### Memo response : The order 1003 has already been delivered.
The good tool was used check_order_status that returned :
'output': '{"format": "CSV", "value": "status,tracking_number,total_amount,estimated_delivery\\ndelivered,TRK1234567890,449.98,Delivered\\n", "truncated": false}'}

## 2 - Log the agent as an MLflow model

Log the agent as code from the `agent.py` file.

### Enable automatic authentication for Databricks resources
For the most common Databricks resource types, Databricks supports and recommends declaring resource dependencies for the agent upfront during logging. This enables automatic authentication passthrough when you deploy the agent. With automatic authentication passthrough, Databricks automatically provisions, rotates, and manages short-lived credentials to securely access these resource dependencies from within the agent endpoint.





In [0]:
# Determine Databricks resources to specify for automatic auth passthrough at deployment time
import mlflow
from pkg_resources import get_distribution

def log_customer_support_agent_model(resources, input_example) :
    # log the agent to the MLFlow Experiment
    # params :
    #    - resources
    #    - input_example 
    
    with mlflow.start_run():
        logged_agent_info = mlflow.pyfunc.log_model(
            name="agent",
            python_model="agent.py",
            extra_pip_requirements=[
                f"databricks-langchain=={get_distribution('databricks-langchain').version}",
                f"langgraph=={get_distribution('langgraph').version}",
                f"databricks-agents=={get_distribution('databricks-agents').version}",
                f"backoff=={get_distribution('backoff').version}",
            ],
        
            resources=resources,
            input_example=input_example,
            model_config="./agent_config.yaml"
        )
    return logged_agent_info

### 3-1 Log the agent

In [0]:
resources = AGENT.get_resources()
print(f"resources {resources}")
logged_agent_info = log_customer_support_agent_model(resources, input_example)

### 3-2 Load and test the logged agent

The agent is now logged mlflow, with its id, we'll load the agent and evaluate before registering.

In [0]:
print(f"Model logged with run_id: {logged_agent_info.run_id}")

In [0]:
import pandas as pd
# Load the model and create a prediction function
class ModelWrapper():
    def __init__(self, run_id) :
        self.loaded_model = mlflow.pyfunc.load_model(f"runs:/{run_id}/agent")

    def predict_wrapper(self, question):
        # Format for chat-style models
        model_input = pd.DataFrame({
            "input": [[{"role": "user", "content": question}]]
        })
        response = self.loaded_model.predict(model_input)
        return response['output'][-1]['content'][-1]['text']

model_wrapper = ModelWrapper(logged_agent_info.run_id)
answer = model_wrapper.predict_wrapper("Give me the status of order 1003.")
print(answer)

## 3- Load and transform an _existing_ evaluation dataset


- load our own evaluation dataset from the repos of the project
- create a table with the dataset in the mlflow format.



In [0]:
import pandas as pd
version=1.
csv_path="https://raw.githubusercontent.com/O-Faraday/databricks_genai_demo/main/data/agent_evaluation_dataset/agent_evaluation_dataset.csv"
"""
  Transform the CSV data into MLflow evaluation dataset format.
"""
df = pd.read_csv(csv_path)
eval_records = []
for _, row in df.iterrows():
    record = {
        "inputs": {
            "question": row["query"],
            },
        "expectations": {
            "expected_facts": [fact.strip() for fact in str(row["key_expected_info"]).split("  ")],
            "expected_retrieved_context": [fact.strip() for fact in str(row["target_data_sources"]).split("  ")]
            },
        "dataset_record_id": str(row["query_id"]),
        "created_by": "oliver",
        "tags": {
            "from": csv_path,
            "dedicated_to": "Databricks",
            "project": "agent_demo",
            "version": version
        }
    }
    eval_records.append(record)

print(f"sample of the data : {eval_records[0]}")
uc_table_name = f"{catalog}.{schema}.agent_evaluation_dataset"
"""
Create an MLflow evaluation dataset in Unity Catalog.
"""
    
print(f"Creating MLflow evaluation dataset: {uc_table_name}")

eval_dataset = mlflow.genai.datasets.create_dataset(
    name=uc_table_name,
    experiment_id="0",  
)

# Add records to the dataset
eval_dataset = eval_dataset.merge_records(eval_records)
    
print(f"✓ Added {len(eval_records)} records to the dataset")




## 4- Evaluate the agent with Agent Evaluation

Use Mosaic AI Agent Evaluation to evaluate the agent's responses based on expected responses and other evaluation criteria. Use the evaluation criteria you specify to guide iterations, using MLflow to track the computed quality metrics.


### 4-1 Get the eval dataset

In [0]:
# Load the eval dataset
uc_table_name = f"{catalog}.{schema}.agent_evaluation_dataset"
eval_dataset = mlflow.genai.datasets.get_dataset(uc_table_name)

### 4-2 Create the scores

In [0]:
import mlflow
from mlflow.genai.scorers import  RelevanceToQuery, Safety, Guidelines, Correctness

def get_scorers():
    return [
        RelevanceToQuery(),  # Checks if response match 
        Correctness(), 
        Safety(),
        Guidelines(
            name="is_written",
            guidelines="This scorer must provide evaluation results as complete, readable sentences written in accessible, high-level language that non-technical stakeholders can understand.",
        ),

    ]

scorers = get_scorers()

### 4-3 Launch the evaluation

In [0]:
with mlflow.start_run(run_name='eval_agent_one'):
    results = mlflow.genai.evaluate(data=eval_dataset, predict_fn=model_wrapper.predict_wrapper, scorers=scorers)

## 5- Improve the agent

### 5-1 Change the system message and the model used

In [0]:
system_prompt = (
    "You are an expert customer support agent with access to tools and knowledge bases.\n"
    "\
    "## YOUR MISSION\n"
    "Help customers efficiently by retrieving accurate information and executing actions through the available tools.\n"
    "\n"
    "## WORKFLOW - Follow this process for EVERY request:\n"
    "\n"
    "### Step 1: UNDERSTAND & PLAN\n"
    "- Analyze the customer's request carefully\n"
    "- Identify what information/actions are needed\n"
    "- Create a brief plan listing the steps you'll take\n"
    "\n"
    "### Step 2: RETRIEVE KNOWLEDGE (When applicable)\n"
    "Use the vector store retriever to get information about:\n"
    "- Return policies (return_policy.pdf)\n"
    "- Technical troubleshooting (technical_faq.pdf)\n"
    "- Product specifications (product_catalog.pdf)\n"
    "- Shipping guidelines (shipping_guide.pdf)\n"
    "\n"
    "**When to retrieve:**\n"
    "- Customer asks about policies, procedures, or guidelines\n"
    "- Technical issues need troubleshooting steps\n"
    "- Product details/specs are needed\n"
    "- Shipping rules or timelines are unclear\n"
    "\n"
    "### Step 3: USE TOOLS (When applicable)\n"
    "Call the appropriate tool functions for:\n"
    "- `check_order_status`: Get order info, tracking, delivery estimates with order_id input type int\n"
    "- `check_product_stock`: Verify product availability with product_id input type int \n"
    "- `get_product_details`: Get pricing, weight, dimensions with product_name type string\n"
    "- `calculate_shipping`: Compute shipping costs\n"
    "- `get_customer_tier_benefits`: Check customer loyalty status with customer id\n"
    "\n"
    "## RESPONSE FORMAT\n"
    "The answer should always be readable sentences written in accessible, high-level language that non-technical stakeholders can understand."
    "Now, help the customer with their request!"
)

In [0]:
import yaml
try:
    config = yaml.safe_load(open("agent_config.yaml"))
    config["config_version_name"] = "anthropic_version"
    config["system_prompt"] = system_prompt
    config["llm_endpoint_name"] = "claude_45"
    yaml.dump(config, open("agent_config.yaml", "w"), default_flow_style=False, allow_unicode=True)
except Exception as e:
    print(f"Skipped update - ignore for job run - {e}")

### 5-2 Load a new agent MLFlow

No need to modify the code

In [0]:
# Let's relog our agent in MLflow to capture the new prompt
import mlflow
from agent import AGENT

logged_agent_info_new = log_customer_support_agent_model(AGENT.get_resources(), input_example)

In [0]:
model_wrapper_new = ModelWrapper(logged_agent_info_new.run_id)
with mlflow.start_run(run_name='eval_with_agent_new'):
    results = mlflow.genai.evaluate(data=eval_dataset, predict_fn=model_wrapper_new.predict_wrapper, scorers=scorers)

## 6- Deploy the agent to production

### 6-1 Pre-deployment agent validation
Before registering and deploying the agent, perform pre-deployment checks using the [mlflow.models.predict()](https://mlflow.org/docs/latest/python_api/mlflow.models.html#mlflow.models.predict) 

In [0]:
mlflow.models.predict(
    model_uri=f"runs:/{logged_agent_info_new.run_id}/agent",
    input_data=input_example,
    env_manager="uv",
)

### 6-2 Register the model to Unity Catalog

Before you deploy the agent, you must register the agent to Unity Catalog.


In [0]:
model_name = "demo-langgraph-agent"
UC_MODEL_NAME = f"{catalog}.{schema}.{model_name}"

# register the model to UC
uc_registered_model_info = mlflow.register_model(
    model_uri=logged_agent_info_new.model_uri, 
    name=UC_MODEL_NAME
)

### 6-3 Deploy the agent

In [0]:
from databricks import agents

agents.deploy(
    UC_MODEL_NAME,
    uc_registered_model_info.version,
    scale_to_zero=True, #Compulsory on the databricks free version
    tags={"endpointSource": "docs"},
)

## 7- Test the agent on the playground